# Colab setup

In [1]:
!pip install -q "gensim" "node2vec" "networkx==3.5" "pecanpy"

In [ ]:
# restart runtime

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# clone repo
!git clone https://github.com/rushikesh-katkar/Entity2vec.git

Cloning into 'Entity2vec'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 27 (delta 5), reused 25 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 82.43 KiB | 4.58 MiB/s, done.
Resolving deltas: 100% (5/5), done.


In [1]:
%cd "/content/Entity2vec/src/main/python"

/content/Entity2vec/src/main/python


In [ ]:
base_path = "/content/drive/MyDrive/Entity2vecData" 
## This is the continuation of the SongEmbeddings
## Need artifacts created there to proceed

# playlist_dict_prunned.pkl
# test_pids.pkl
# eval_dict.pkl
# cooc_matrix.npz

# Data

In [ ]:
%cd "/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python/"

/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python


# Tokenizer

In [3]:
import torch
from utils.Tokenizer.Tokenizer import simpleTokenizer

In [ ]:
import pickle

load_path = base_path + "/playlist_dict_prunned.pkl"


with open(load_path, "rb") as f:
    playlist_dict_new = pickle.load(f)

print("Loaded playlists:", len(playlist_dict_new))

Loaded playlists: 1000000


In [5]:
tokenizer = simpleTokenizer(playlist_dict_new)

In [6]:
len(tokenizer.itos)

70229

In [ ]:
freq = {}

for ls in playlist_dict_new.values():

    for _ in ls:

        freq[tokenizer.stoi[_]] = freq.get(tokenizer.stoi[_], 0) + 1



In [ ]:
import numpy as np

vals = np.array(list(freq.values()))

print("tokens       :", len(vals))
print("total_count  :", vals.sum())
print("mean         :", vals.mean())
print("std          :", vals.std())
print("min          :", vals.min())
print("25%          :", np.percentile(vals, 25))
print("median       :", np.median(vals))
print("75%          :", np.percentile(vals, 75))
print("max          :", vals.max())

tokens       : 70229
total_count  : 53521564
mean         : 762.1006137065884
std          : 1905.8007094890263
min          : 100
25%          : 144.0
median       : 240.0
75%          : 547.0
max          : 46574


In [7]:
from gensim.models import Word2Vec
from node2vec import Node2Vec
import networkx as nx
import numpy as np
from collections import defaultdict
from tqdm import tqdm


In [8]:
import pickle

with open(base_path + "/test_pids.pkl", "rb") as f:
    test_ids = set(pickle.load(f))

train_playlists = {
    pid: tracks for pid, tracks in playlist_dict_new.items()
    if pid not in test_ids
}

test_playlists = {
    pid: tracks for pid, tracks in playlist_dict_new.items()
    if pid in test_ids
}

In [ ]:
def build_corpus(playlist_dict, tokenizer):
    """Convert playlist dict to gensim-compatible corpus."""
    corpus = []
    for pid, track_ids in tqdm(playlist_dict.items()):
        # convert track_ids to string tokens (gensim needs strings)
        tokens = [str(tokenizer.stoi[t]) for t in track_ids if t in tokenizer.stoi]
        if len(tokens) > 1:
            corpus.append(tokens)
    return corpus

corpus = build_corpus(train_playlists, tokenizer)
print(f"Total playlists (sentences): {len(corpus)}")
print(f"Sample: {corpus[0][:5]}")

100%|██████████| 950000/950000 [01:38<00:00, 9626.63it/s]  

Total playlists (sentences): 934947
Sample: ['0', '1', '2', '3', '4']


In [ ]:
from gensim.models.callbacks import CallbackAny2Vec

class EpochLogger(CallbackAny2Vec):
    def __init__(self):
        self.epoch = 0
    def on_epoch_end(self, model):
        self.epoch += 1
        print(f"Epoch {self.epoch} complete")

w2v_model = Word2Vec(
    sentences=corpus,
    vector_size=128,
    window=5,
    min_count=1,
    sg=1,
    workers=2,
    epochs=5,
    callbacks=[EpochLogger()]
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Epoch 1 complete


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Epoch 2 complete


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Epoch 3 complete


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Epoch 4 complete


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Epoch 5 complete


In [ ]:
# w2v_model.save(base_path + "w2v_song_embeddings.model")

# load back
w2v_model = Word2Vec.load(base_path + "/w2v_song_embeddings.model")

# Evaluation W2V

In [ ]:
all_tracks = set(tokenizer.stoi.keys())
playlist_tracks = set(t for tracks in train_playlists.values() for t in tracks)

missing = all_tracks - playlist_tracks
print(f"Tracks in tokenizer but not in playlists: {len(missing)}")

Tracks in tokenizer but not in playlists: 0


In [ ]:
del all_tracks, playlist_tracks, missing

In [ ]:
# First Normalize the vectors
vectors = w2v_model.wv.vectors
w2v_model.wv.vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

In [ ]:
v1 = w2v_model.wv[tokenizer.stoi[test_playlists[549005][0]]]
v2 = w2v_model.wv[tokenizer.stoi[test_playlists[549005][1]]]
v1 @ v2
ct = 0
for i in v1:
    ct += i**2
ct

0.9999999836842335

In [ ]:
v1 @ v2

0.55068016

In [ ]:
import pickle

# load eval_dict
with open(base_path + "/eval_dict.pkl", "rb") as f:
    eval_dict = pickle.load(f)
print(f"eval_dict loaded! Size: {len(eval_dict)}")

46929


In [ ]:
import numpy as np

def evaluate_embeddings(X, eval_dict, k_recall=2):
    recalls = []
    ranks = []

    for ex in tqdm(eval_dict.values()):
        pivot = ex["pivot"]
        positives = ex["positives"]
        negatives = ex["negatives"]

        candidates = positives + negatives

        pivot_vec = X[str(int(pivot))]

        sims = [pivot_vec @ X[str(int(t))] for t in candidates]

        # rank (descending similarity)
        order = np.argsort(sims)[::-1]

        # positions of positives
        pos_ranks = []
        for idx in range(len(positives)):
            rank = np.where(order == idx)[0][0] + 1  # 1-based
            pos_ranks.append(rank)

        # Recall@k
        recall = sum(r <= k_recall for r in pos_ranks) / len(pos_ranks)
        recalls.append(recall)

        # Average rank
        ranks.extend(pos_ranks)

    return {
        "recall@{}".format(k_recall): np.mean(recalls),
        "avg_rank": np.mean(ranks),
    }

In [23]:
# metrics = evaluate_embeddings(w2v_model.wv, eval_dict, k_recall=2)
# print(metrics)

# Node 2 Vec

In [ ]:
from scipy.sparse import load_npz

cooc = load_npz(base_path + "/cooc_matrix.npz")

print(f"Filled : {100*cooc.nnz/(cooc.shape[0] * cooc.shape[1]):.2f}%")

Filled : 2.93%


In [ ]:

# Postitive pointwise mutual information
import numpy as np
from scipy.sparse import csr_matrix

def csr_to_ppmi(cooc: csr_matrix):
    cooc = cooc.tocsr().astype(np.float64)

    # totals
    total = cooc.sum()
    row_sum = np.array(cooc.sum(axis=1)).flatten()
    col_sum = np.array(cooc.sum(axis=0)).flatten()

    # avoid divide-by-zero
    row_sum[row_sum == 0] = 1
    col_sum[col_sum == 0] = 1

    # iterate over nonzeros only
    rows, cols = cooc.nonzero()
    data = cooc.data

    # PMI
    pmi = np.log((data * total) / (row_sum[rows] * col_sum[cols]))

    # PPMI
    pmi[pmi < 0] = 0

    return csr_matrix((pmi, (rows, cols)), shape=cooc.shape)

ppmi = csr_to_ppmi(cooc)

In [ ]:
# Cell 2 - imports
import numpy as np
import gc
import warnings
from pecanpy import pecanpy as pca
from gensim.models import Word2Vec


In [ ]:

# Cell 3 - write edgelist from ppmi (chunked to save RAM)
ppmi_csr = ppmi.tocsr().astype(np.float32)
ppmi_csr.eliminate_zeros()
del ppmi
gc.collect()


In [ ]:
from tqdm import tqdm

chunk_size = 1000
total_chunks = (ppmi_csr.shape[0] + chunk_size - 1) // chunk_size

with open(base_path + "/graph.edgelist", "w") as f:
    for i in tqdm(range(0, ppmi_csr.shape[0], chunk_size), total=total_chunks, desc="Writing edgelist"):
        chunk = ppmi_csr[i:i+chunk_size].tocoo()
        for u, v, w in zip(chunk.row + i, chunk.col, chunk.data):
            if u < v:
                f.write(f"{u} {v} {w}\n")
        del chunk
        gc.collect()

del ppmi_csr
gc.collect()


Writing edgelist: 100%|██████████| 71/71 [03:38<00:00,  3.08s/it]


0

In [22]:
with open(base_path + "/graph.edgelist", "r") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i == 4:
            break

0 1 5.137670040130615
0 2 2.164637565612793
0 3 7.039037704467773
0 4 5.297184467315674
0 5 6.918213367462158


In [6]:
# Cell 4 - load into pecanpy
import numpy as np
import gc
import warnings
from pecanpy import pecanpy as pca
g = pca.SparseOTF(p=2, q=0.5, workers=2, verbose=True)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    g.read_edg(base_path + "/graph.edgelist", weighted=True, directed=False, delimiter=" ")


In [7]:
# Cell 5 -  simulate walks
walks = g.simulate_walks(num_walks= 10, walk_length = 100)

  0%|          | 0/702290 [00:00<?, ?it/s]

In [8]:

# save walks immediately
import pickle
with open(base_path + "/walks.pkl", "wb") as f:
    pickle.dump(walks, f)
print("Walks saved!")


Walks saved!


In [9]:
import pickle

with open(base_path + "/walks.pkl", "rb") as f:
    walks = pickle.load(f)
print(f"Walks loaded! Total walks: {len(walks)}")

Walks loaded! Total walks: 702290


In [10]:
len(set(walks[0]))

101

In [11]:
# Cell 6 - train Word2Vec¯
n2v_model = Word2Vec(
    walks,
    vector_size=128,
    window=15,
    min_count=1,
    negative=5,
    workers=2,
    epochs=5
)

del walks
# gc.collect()


NameError: name 'gc' is not defined

In [12]:

# save model immediately
n2v_model.save(base_path + "/n2v_model")
print("Model saved!")

Model saved!


In [ ]:
#

# Evaluation N2V

In [ ]:
import numpy as np
from tqdm import tqdm

In [13]:
from gensim.models import Word2Vec

n2v_model = Word2Vec.load(base_path + "/n2v_model")
print("Model loaded!")
print(f"Vocabulary size: {len(n2v_model.wv)}")

Model loaded!
Vocabulary size: 70229


In [14]:
all_tracks = set(tokenizer.stoi.keys())
playlist_tracks = set(t for tracks in train_playlists.values() for t in tracks)

missing = all_tracks - playlist_tracks
print(f"Tracks in tokenizer but not in playlists: {len(missing)}")

Tracks in tokenizer but not in playlists: 0


In [15]:
del all_tracks, playlist_tracks, missing

In [16]:
# First Normalize the vectors
vectors = n2v_model.wv.vectors
n2v_model.wv.vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

In [17]:
v1 = n2v_model.wv[tokenizer.stoi[test_playlists[549005][0]]]
v2 = n2v_model.wv[tokenizer.stoi[test_playlists[549005][1]]]
v1 @ v2
ct = 0
for i in v1:
    ct += i**2
ct

1.0000000833003386

In [18]:
v1 @ v2

0.4252728

In [19]:
import pickle

# load eval_dict
with open(base_path + "/eval_dict.pkl", "rb") as f:
    eval_dict = pickle.load(f)
print(f"eval_dict loaded! Size: {len(eval_dict)}")

eval_dict loaded! Size: 46929


In [20]:
import numpy as np

def evaluate_embeddings(X, eval_dict, k_recall=2):
    recalls = []
    ranks = []

    for ex in tqdm(eval_dict.values()):
        pivot = ex["pivot"]
        positives = ex["positives"]
        negatives = ex["negatives"]

        candidates = positives + negatives

        pivot_vec = X[str(int(pivot))]

        sims = [pivot_vec @ X[str(int(t))] for t in candidates]

        # rank (descending similarity)
        order = np.argsort(sims)[::-1]

        # positions of positives
        pos_ranks = []
        for idx in range(len(positives)):
            rank = np.where(order == idx)[0][0] + 1  # 1-based
            pos_ranks.append(rank)

        # Recall@k
        recall = sum(r <= k_recall for r in pos_ranks) / len(pos_ranks)
        recalls.append(recall)

        # Average rank
        ranks.extend(pos_ranks)

    return {
        "recall@{}".format(k_recall): np.mean(recalls),
        "avg_rank": np.mean(ranks),
    }

In [21]:
metrics = evaluate_embeddings(n2v_model.wv, eval_dict, k_recall=2)
print(metrics)

100%|██████████| 46929/46929 [00:04<00:00, 11474.73it/s]

{'recall@2': 0.7570372264484647, 'avg_rank': 2.237006967972895}
